# Distinct sums

Take a list of **r** numbers. Add up exactly **n** of them, reusing any value as often as you like. This counts how many different totals you can reach.

Order does not matter. From the list 1, 2, 3 with n = 4, both 1+3+2+1 and 1+1+2+3 come to 7, so 7 is counted once.

<br>

### Running it

Press **Setup** once. Then set **r**, set **n**, and press **Run**.

After changing the formula, press **r** again before running.

<br>

### The list

Entry number $j$ of the list comes from the **Formula** box. It starts as

$$2\cos\!\left(\frac{\pi j}{r+1}\right)$$

so entry 1 uses $j = 1$, entry 2 uses $j = 2$, on up to entry $r$. Type any formula you like in its place. The symbols you can use are listed under the box.

<br>

### Rounding

**ndigits** is the decimal place totals are rounded to before they are compared, so it decides which totals count as the same one.

Its slider stops at the finest setting this machine's memory can hold, which depends on r and n.

Move it and run again. A count that holds steady across several settings is the true one. A count that keeps changing means the setting is still too coarse.

<br>

For a large r, set the runtime to a GPU.

In [ ]:
#@title Setup { display-mode: "form" }
!test -d permute-on-r-and-n || git clone -q https://github.com/AbbasCherri/permute-on-r-and-n.git
import math
import sys
import ipywidgets as widgets
from IPython.display import display
from time import perf_counter
from tqdm.auto import tqdm
sys.path[:0] = ['permute-on-r-and-n', '.']
from permutations import count_distinct_sums, max_ndigits, free_memory, gpu_ready

SYMBOLS = {name: getattr(math, name) for name in dir(math) if not name.startswith('_')}
SYMBOLS.update(abs=abs, round=round, min=min, max=max, __builtins__={})
print(f"ready on {'GPU' if gpu_ready() else 'CPU'}, {free_memory() >> 20} MiB free")

In [ ]:
#@title Formula { display-mode: "form" }
formula = "2*cos(pi*j/(r+1))"  #@param {type:"string"}
try:
    entry = eval('lambda j, r: ' + formula.replace('^', '**'), SYMBOLS)
    print(f'entry j  =  {formula}')
except SyntaxError:
    entry = None
    print(f'cannot read "{formula}", check it against the symbols below')
print("""
  j          which entry, counting 1, 2, 3, ... up to r
  r          how many entries there are

  pi         3.14159...            e          2.71828...
  +  -       add, subtract         *  /       multiply, divide
  ^          power                 sqrt(x)    square root
  abs(x)     size, ignoring sign   factorial(x)
  exp(x)     e to the power x      log(x)     natural log, log10(x) for base 10
  cos(x)  sin(x)  tan(x)  acos(x)  asin(x)  atan(x)   angles in radians

  write every multiplication out:  2*cos(...)  not  2cos(...)""")

In [ ]:
#@title r { display-mode: "form" }
r = 4  #@param {type:"integer"}
if entry is None:
    print('the formula could not be read, fix it in the Formula box first')
else:
    t = perf_counter()
    values = [entry(j, r) for j in range(1, r + 1)]
    print(f'{len(values)} values in {perf_counter() - t:.3f}s')

In [ ]:
#@title n { display-mode: "form" }
n = 10  #@param {type:"integer"}
finest = max_ndigits(values, n)
ndigits = widgets.IntSlider(value=min(9, finest), min=0, max=finest,
                            description='ndigits', continuous_update=False)
display(ndigits)

In [ ]:
#@title Run { display-mode: "form" }
t = perf_counter()
count = count_distinct_sums(values, n, ndigits=ndigits.value, progress=tqdm)
print(f'{count} distinct sums at ndigits={ndigits.value} in {perf_counter() - t:.3f}s')